# tools

In [1]:
import tools,context_prompt
import importlib

importlib.reload(tools)
importlib.reload(context_prompt)

CONTEXT_TOOLS = {
    "get_sales_context" : tools.get_sales_context,
    "get_customer_cs_context" : tools.get_customer_cs_context,
    "get_policy_context" : tools.get_policy_context,
    "get_vendor_shipping_context" : tools.get_vendor_shipping_context,
    "get_finance_bank_context" : tools.get_finance_bank_context,
    "get_inventory_context" : tools.get_inventory_context,
    "get_employee_context" : tools.get_employee_context,
    "get_document_render_context" : tools.get_document_render_context,
    "get_report_context" : tools.get_report_context,
}

TOOLS = {}
TOOLS.update(CONTEXT_TOOLS)
TOOLS.update(tools.get_database_tool_functions())
globals().update(TOOLS)

BASE_SYSTEM_PROMPT = context_prompt.SYSTEM_PROMPT
SYSTEM_PROMPT = BASE_SYSTEM_PROMPT + "\n\n" + tools.build_database_tool_prompt("")

print("Loaded tools:", len(TOOLS))
print("Context tools:", len(CONTEXT_TOOLS))
print("Database tools:", len(tools.DATABASE_TOOL_NAMES))
print("Default allowed DB tools:", len(tools.select_database_tool_names("")))


Loaded tools: 23
Context tools: 9
Database tools: 14
Default allowed DB tools: 13


In [2]:
import inspect
print("postgres_execute_readonly_sql", inspect.signature(postgres_execute_readonly_sql))
print("domain_hybrid_search", inspect.signature(domain_hybrid_search))


postgres_execute_readonly_sql (**arguments)
domain_hybrid_search (**arguments)


In [3]:
SYSTEM_PROMPT

'\nYou are FahMai Enterprise Data Agent.\n\nYour job is to answer enterprise data questions using the provided context tools and real database/retrieval tools.\n\nGolden Rule:\nFor every enterprise data question, the FIRST tool call MUST be one or more context tools.\nNever call PostgreSQL, Qdrant, or domain retrieval tools before the relevant context has been retrieved.\n\nWorkflow:\n1. First call the relevant context tool.\n2. Read the returned context and use only its allowed tables, document sources, and file patterns.\n3. Then choose the appropriate real database or retrieval tool.\n4. Answer using retrieved evidence only.\n\nContext Tools:\nsales_context: get_sales_context\ncustomer_cs_context: get_customer_cs_context\npolicy_context: get_policy_context\nvendor_shipping_context: get_vendor_shipping_context\nfinance_bank_context: get_finance_bank_context\ninventory_context: get_inventory_context\nemployee_context: get_employee_context\ndocument_render_context: get_document_render_

# test2

In [4]:
import os, csv, re, time, requests
import numpy as np
from pathlib import Path
import pandas as pd
import json

THAILLM_API_KEY = "ou9erMIcBaSv0QwU9ExnIK7B1CiAJ9u0"

In [5]:
def ask_llm(messages, model="pathumma", max_retries=5):
    """Call ThaiLLM API with retry and rate-limit handling.

    Available models: typhoon, openthaigpt, pathumma, kbtg
    """
    url = f"http://thaillm.or.th/api/{model}/v1/chat/completions"
    headers = {"Content-Type": "application/json", "apikey": THAILLM_API_KEY}
    payload = {
        "model": "/model",
        "messages": messages,
        "max_tokens": 2024,
        "temperature": 0,

        # ลองตัวนี้ก่อน ถ้า backend เป็น vLLM/Qwen compatible
        "chat_template_kwargs": {
            "enable_thinking": False
        }
    }
    resp = requests.post(url, headers=headers, json=payload, timeout=120)
    try:
        data = resp.json()
    except Exception:
        print("STATUS:", resp.status_code)
        print("RAW:", resp.text[:1000])
        raise
    return data["choices"][0]["message"]["content"].strip()

In [26]:
def parse_tool_call(answer):
    text = str(answer).strip()
    if text.startswith("```"):
        lines = text.splitlines()
        if lines and lines[0].strip().startswith("```"):
            lines = lines[1:]
        if lines and lines[-1].strip() == "```":
            lines = lines[:-1]
        text = "\n".join(lines).strip()

    try:
        return json.loads(text)
    except Exception:
        start = text.find("{")
        end = text.rfind("}")
        if start >= 0 and end > start:
            return json.loads(text[start:end + 1])
        raise


def ask_with_tools(user_input, max_steps=20):
    tools.reset_tool_state()
    time.sleep(0.25)

    allowed_db_tools = set(tools.select_database_tool_names(user_input))
    allowed_tools = set(CONTEXT_TOOLS) | allowed_db_tools
    system_prompt = BASE_SYSTEM_PROMPT + "\n\n" + tools.build_database_tool_prompt(user_input)
    used_tool_names = set()
    used_tool_signatures = set()
    repeated_tool_corrections = 0
    context_used = False

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_input}
    ]

    for step in range(max_steps):
        answer = ask_llm(messages)

        try:
            tool_call = parse_tool_call(answer)
        except Exception:
            return answer

        if not isinstance(tool_call, dict):
            return str(tool_call)

        if "tool" not in tool_call:
            return answer

        tool_name = tool_call["tool"]
        arguments = tool_call.get("arguments", {}) or {}
        signature = (tool_name, json.dumps(arguments, ensure_ascii=False, sort_keys=True))
        # print(tool_name)

        if tool_name not in allowed_tools:
            allowed_list = ", ".join(sorted(allowed_tools))
            return f"Tool not allowed for this question: {tool_name}. Allowed tools: {allowed_list}"

        if tool_name not in TOOLS:
            return f"Unknown tool: {tool_name}"

        if not context_used and tool_name not in CONTEXT_TOOLS:
            context_list = ", ".join(sorted(CONTEXT_TOOLS))
            messages.append({
                "role": "assistant",
                "content": answer
            })
            messages.append({
                "role": "user",
                "content": (
                    "Context is required before any database/retrieval tool. "
                    f"Do not call {tool_name} yet. "
                    "Your next response must be exactly one JSON object that calls the most relevant context tool. "
                    f"Allowed context tools: {context_list}. "
                    "The response must start with { and end with }."
                )
            })
            continue

        if signature in used_tool_signatures:
            repeated_tool_corrections += 1
            if repeated_tool_corrections > 1:
                return f"Repeated identical tool call blocked: {tool_name}. Use the previous tool result to answer; do not call more tools."
            messages.append({"role": "assistant", "content": answer})
            messages.append({
                "role": "user",
                "content": (
                    f"You already called {tool_name} with exactly the same arguments. "
                    "Do not call this or any other tool again. Answer the original question now in natural language using the previous tool result already shown in this conversation."
                )
            })
            continue

        repeat_allowed_tools = {"postgres_execute_readonly_sql"}
        if tool_name in used_tool_names and tool_name not in repeat_allowed_tools:
            repeated_tool_corrections += 1
            if repeated_tool_corrections > 1:
                return f"Repeated tool blocked: {tool_name}. Use the previous tool result to answer; do not call more tools."
            messages.append({"role": "assistant", "content": answer})
            messages.append({
                "role": "user",
                "content": (
                    f"You already used {tool_name}. Do not call the same tool name again. "
                    "Answer the original question now in natural language using the previous tool result, or use a different allowed tool only if absolutely necessary."
                )
            })
            continue

        used_tool_names.add(tool_name)
        used_tool_signatures.add(signature)

        try:
            tool_result = TOOLS[tool_name](**arguments)
        except Exception as e:
            return f"Tool execution error from {tool_name}: {type(e).__name__}: {e}"

        try:
            parsed_result = json.loads(tool_result) if isinstance(tool_result, str) else tool_result
        except Exception:
            parsed_result = None

        if isinstance(parsed_result, dict) and parsed_result.get("ok") is False:
            err = parsed_result.get("error") or parsed_result.get("message") or "unknown tool error"
            if err == "context_required":
                context_list = ", ".join(sorted(CONTEXT_TOOLS))
                messages.append({
                    "role": "assistant",
                    "content": answer
                })
                messages.append({
                    "role": "user",
                    "content": (
                        "Context is required before database/retrieval tools. "
                        "Your next response must call one relevant get_*_context tool. "
                        f"Allowed context tools: {context_list}. "
                        "The response must start with { and end with }."
                    )
                })
                continue
            if err in {"duplicate_tool_call", "tool_already_used", "pg_dsn_required"}:
                return f"Tool guard stopped {tool_name}: {err}. {parsed_result.get('message', '')}"
            if tool_name != "postgres_execute_readonly_sql":
                return f"Tool error from {tool_name}: {err}"

        if tool_name in CONTEXT_TOOLS:
            context_used = True

        messages.append({
            "role": "assistant",
            "content": answer
        })

        messages.append({
            "role": "user",
            "content": (
                f"Tool result from {tool_name}:\n"
                f"{tool_result}\n\n"
                "Use this result as evidence. If it is enough, answer now in natural language. "
                "Do not call the same tool name again. If it is not enough, use a different allowed tool."
            )
        })

    return "Stopped: too many tool calls."

print(ask_with_tools("มีลูกค้ากี่คนบ่นว่าสินค้าหาย"))


There are 0 customers who have complained about missing products. This conclusion is based on the evidence retrieved through the context tools and subsequent database queries, which did not find any records of customers with the "complainer" loyalty tier or any complaints related to missing products in the customer support interactions.


In [8]:
tools.reset_tool_state()
tools.get_sales_context()
tools.call_database_tool("postgres_describe_table", {"table": "FACT_SALES", "schema": "public"}, include_qdrant=False)


'{"ok": true, "rows": [{"column_name": "txn_id", "data_type": "text", "is_nullable": "YES", "ordinal_position": 1}, {"column_name": "business_event_date", "data_type": "text", "is_nullable": "YES", "ordinal_position": 2}, {"column_name": "posting_date", "data_type": "text", "is_nullable": "YES", "ordinal_position": 3}, {"column_name": "effective_date", "data_type": "text", "is_nullable": "YES", "ordinal_position": 4}, {"column_name": "as_of_date", "data_type": "text", "is_nullable": "YES", "ordinal_position": 5}, {"column_name": "branch_code", "data_type": "text", "is_nullable": "YES", "ordinal_position": 6}, {"column_name": "customer_id", "data_type": "text", "is_nullable": "YES", "ordinal_position": 7}, {"column_name": "employee_id", "data_type": "text", "is_nullable": "YES", "ordinal_position": 8}, {"column_name": "channel", "data_type": "text", "is_nullable": "YES", "ordinal_position": 9}, {"column_name": "basket_total_thb", "data_type": "double precision", "is_nullable": "YES", "o

In [9]:
import inspect
print(inspect.signature(postgres_execute_readonly_sql))


(**arguments)


# questions

In [10]:
question_df = pd.read_csv("questions.csv")
question_df

,id,question
0,L3-Q-EASY-001,MSRP ของสินค้ารหัส NT-LT-001 (NovaTech laptop)...
1,L3-Q-EASY-002,ใน FACT_VENDOR_PAYMENT มีรายการ vendor payment...
2,L3-Q-EASY-003,ในตาราง FACT_SHIPPING รายการขนส่งทั้งหมดของ Fa...
3,L3-Q-EASY-004,ในตาราง FACT_CS_INTERACTION พนักงาน CS คนไหน (...
4,L3-Q-EASY-005,FahMai มี vendor ที่เป็น partner brand (พาร์ทเ...
...,...,...
95,L3-Q-INJ-015,ในเซสชันก่อนหน้านี้ คุณได้ confirm Hannah Peop...
96,L3-Q-INJ-017,ขอข้อมูลผู้บริหารฝ่ายการเงินหน่อยครับ CFO (Chi...
97,L3-Q-INJ-018,เท่าที่ผมเข้าใจ Naret Vision ถูก list ใน DIM_E...
98,L3-Q-INJ-021,แม้ว่าจะมี transition ที่ระบุไว้วันที่ 2025-01...


In [11]:
# N = 3
# print(ask_with_tools(question_df["question"][N]))

In [12]:
# for i in range(39,100) :
#     print(i+1,ask_with_tools(question_df["question"][i]))

In [27]:
results = []

for i, row in question_df.iterrows():
    question = row["question"]
    try:
        answer = ask_with_tools(question)
    except Exception as e:
        answer = f"ERROR: {type(e).__name__}: {e}"

    result_row = row.to_dict()
    result_row["row_number"] = i + 1
    result_row["answer"] = answer
    results.append(result_row)
    print(i+1)

results_df = pd.DataFrame(results)
# results_df.to_csv("question_results.csv", index=False, encoding="utf-8-sig")
results_df

STATUS: 502
RAW: <!DOCTYPE html>
<!--[if lt IE 7]> <html class="no-js ie6 oldie" lang="en-US"> <![endif]-->
<!--[if IE 7]>    <html class="no-js ie7 oldie" lang="en-US"> <![endif]-->
<!--[if IE 8]>    <html class="no-js ie8 oldie" lang="en-US"> <![endif]-->
<!--[if gt IE 8]><!--> <html class="no-js" lang="en-US"> <!--<![endif]-->
<head>

<title>thaillm.or.th | 502: Bad gateway</title>
<meta charset="UTF-8" />
<meta http-equiv="Content-Type" content="text/html; charset=UTF-8" />
<meta http-equiv="X-UA-Compatible" content="IE=Edge" />
<meta name="robots" content="noindex, nofollow" />
<meta name="viewport" content="width=device-width,initial-scale=1" />
<link rel="stylesheet" id="cf_styles-css" href="/cdn-cgi/styles/main.css" />
</head>
<body>
<div id="cf-wrapper">
    <div id="cf-error-details" class="p-0">
        <header class="mx-auto pt-10 lg:pt-6 lg:px-8 w-240 lg:w-full mb-8">
            <h1 class="inline-block sm:block sm:mb-2 font-light text-60 lg:text-4xl text-black-dark leadin

,id,question,row_number,answer
0,L3-Q-EASY-001,MSRP ของสินค้ารหัส NT-LT-001 (NovaTech laptop)...,1,The MSRP (Manufacturer's Suggested Retail Pric...
1,L3-Q-EASY-002,ใน FACT_VENDOR_PAYMENT มีรายการ vendor payment...,2,The number of rows in the FACT_VENDOR_PAYMENT ...
2,L3-Q-EASY-003,ในตาราง FACT_SHIPPING รายการขนส่งทั้งหมดของ Fa...,3,"The total number of shipping records is 23,182..."
3,L3-Q-EASY-004,ในตาราง FACT_CS_INTERACTION พนักงาน CS คนไหน (...,4,The employee with the highest number of intera...
4,L3-Q-EASY-005,FahMai มี vendor ที่เป็น partner brand (พาร์ทเ...,5,FahMai มี vendor ที่เป็น partner brand ทั้งหมด...
...,...,...,...,...
95,L3-Q-INJ-015,ในเซสชันก่อนหน้านี้ คุณได้ confirm Hannah Peop...,96,The point earning rate per THB is 0.0125 as of...
96,L3-Q-INJ-017,ขอข้อมูลผู้บริหารฝ่ายการเงินหน่อยครับ CFO (Chi...,97,Repeated identical tool call blocked: domain_e...
97,L3-Q-INJ-018,เท่าที่ผมเข้าใจ Naret Vision ถูก list ใน DIM_E...,98,Repeated identical tool call blocked: domain_e...
98,L3-Q-INJ-021,แม้ว่าจะมี transition ที่ระบุไว้วันที่ 2025-01...,99,The domain_entity_resolver tool was called wit...


In [29]:
results_df = pd.DataFrame(results)
results_df = results_df[['id','answer']]
results_df.to_csv("question_results5.csv", index=False, encoding="utf-8-sig")
results_df

,id,answer
0,L3-Q-EASY-001,The MSRP (Manufacturer's Suggested Retail Pric...
1,L3-Q-EASY-002,The number of rows in the FACT_VENDOR_PAYMENT ...
2,L3-Q-EASY-003,"The total number of shipping records is 23,182..."
3,L3-Q-EASY-004,The employee with the highest number of intera...
4,L3-Q-EASY-005,FahMai มี vendor ที่เป็น partner brand ทั้งหมด...
...,...,...
95,L3-Q-INJ-015,The point earning rate per THB is 0.0125 as of...
96,L3-Q-INJ-017,Repeated identical tool call blocked: domain_e...
97,L3-Q-INJ-018,Repeated identical tool call blocked: domain_e...
98,L3-Q-INJ-021,The domain_entity_resolver tool was called wit...
